# Volatility-managed factors

Factor investing usually begins by asking which return sources to hold. Volatility management instead changes **how much exposure to take to the same factor over time**. Moreira and Muir (2017) propose reducing factor exposure after high realized variance and increasing it after calm months. 

In this notebook, I first outline the argument for volatility management and then apply it to the Fama–French factors and momentum. I focus on a strategy that adjusts exposure inversely to realized variance in the preceding month. I then examine whether the managed returns can be explained by constant exposures to the original factors, with particular attention to volatility-managed momentum. The analysis uses Kenneth French's daily and monthly returns, using a ten-year calibration window followed by a common out-of-sample evaluation.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from factor_asset_pricing.french_data import load_french
from factor_asset_pricing.asset_pricing_tests import sharpe_ratio, time_series_regression
from factor_asset_pricing.volatility_management import aggregate_realized_variance, calibrate_volatility_management

RAW = ROOT / 'data/raw/french'
NAVY, BLUE, TEAL, AMBER, GRAY = '#17324D', '#2878A5', '#168A86', '#D99A32', '#737D87'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11, 'axes.titlesize': 12,
    'axes.labelsize': 11, 'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'figure.dpi': 110, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False, 'grid.alpha': .15,
    'lines.linewidth': 1.8})

## 1. The economic intuition behind volatility management

The idea in Moreira and Muir (2017) is to reduce exposure when recent realized variance is high and increase it when variance is low. The intuition is that volatility tends to cluster: large return movements often follow other large movements, so the previous month's variance provides some information about risk in the next month.

Maintaining the same exposure through both calm and turbulent periods may be unattractive if higher risk is not accompanied by sufficiently higher expected returns. Volatility management addresses this by adjusting the size of the position, rather than changing which factor is held or attempting to predict the direction of its next return.

The strategy therefore relies on recent volatility being informative about future risk and expected returns not increasing proportionately with variance. Neither condition is guaranteed across factors or periods. Here, I use the previous month's realized variance as a simple, observable measure of risk rather than introducing a separate forecasting model.


### 1.1 Volatility-management strategy

The volatility-management strategy follows three steps: estimate recent risk, determine exposure for the following month, and apply that exposure to the factor return.

#### Measuring realized variance

First, realized variance is calculated from daily factor returns within each month:

$$
RV_t=\sum_{d\in t}(F_d-\bar F_t)^2.
$$

Here, $F_d$ denotes the daily factor return and $\bar F_t$ its average within month $t$. Squaring the deviations gives greater weight to large return movements. The resulting measure is neither divided by the number of trading days nor annualized, and daily and monthly returns are expressed in consistent units.

#### Determining factor exposure

The position for month $t$ is inversely proportional to the preceding month's realized variance, consistent with mean-variance portfolio choice:

$$
w_t=\frac{c}{RV_{t-1}}.
$$

Assuming approximately constant expected returns, higher realized variance reduces exposure, while the constant $c$ determines the overall scale of the position according to the investor's risk preferences. Moreira and Muir (2017) choose $c$ ex post to match the original factor's unconditional volatility:

$$
c=\frac{\operatorname{SD}(F_t)}{\operatorname{SD}(F_t/RV_{t-1})}.
$$

However, since investors must determine $c$ ex ante, our analysis in the example below instead calibrates it using an initial 10-year window to match the factor's historical volatility and holds it fixed out of sample. This preserves comparability without introducing look-ahead bias.

#### Constructing managed returns

The volatility-managed factor return is then

$$
F_t^{VM}=w_tF_t,
$$

where $F_t$ denotes the original monthly factor return. Since the weight is always positive, the strategy adjusts exposure without reversing the underlying position. Weights above one imply leveraged exposure, while weights below one reduce exposure. Because the factors represent excess-return payoffs, the risk-free return is not subtracted again.

## 2. Example: Volatility-managing the Fama–French factors and momentum

I apply the rule to MKT, SMB, HML, RMW, CMA and MOM using daily and monthly French returns. Daily returns measure realized variance, while monthly returns supply the holding-period payoff. MKT, SMB and HML use FF3 sources; RMW and CMA use FF5 sources.

The first 120 jointly usable months calibrate one fixed constant per factor. All performance comparisons then use the same subsequent months, through July 2026. The constants are never refitted, and the charts use these same positions without further volatility matching. The source histories contain revisions, so this is an out-of-sample timing exercise with the available data, rather than a reconstruction of historical data vintages.

In [ ]:
END = '2026-07'
CALIBRATION_MONTHS = 120
rename3 = {'MKT': 'mkt_rf', 'SMB': 'smb', 'HML': 'hml', 'RF': 'rf'}
rename5 = {'MKT': 'ff5_mkt_rf', 'SMB': 'ff5_smb', 'HML': 'ff5_hml',
           'RMW': 'rmw', 'CMA': 'cma', 'RF': 'ff5_rf'}

def public_factors(frequency):
    ff3 = load_french('ff3', RAW, frequency=frequency).rename(columns=rename3)
    ff5 = load_french('ff5', RAW, frequency=frequency).rename(columns=rename5)
    mom = load_french('momentum', RAW, frequency=frequency).rename(columns={'MOM': 'mom'})
    return pd.concat([ff3, ff5, mom], axis=1).sort_index().loc[:END]

daily = public_factors('daily')
data = public_factors('monthly')
data = data.reindex(pd.date_range(data.index.min(), data.index.max(), freq='MS', name='date'))
columns = {'MKT': 'mkt_rf', 'SMB': 'smb', 'HML': 'hml', 'RMW': 'rmw', 'CMA': 'cma', 'MOM': 'mom'}
rv = aggregate_realized_variance(daily.reset_index(), return_cols=list(columns.values()),
                                 demean=True, min_obs=2).set_index('period')
rv.index = rv.index.to_timestamp()
original_history = pd.DataFrame({name: data[column] for name, column in columns.items()})
variance_history = pd.DataFrame({name: rv[f'rv_{column}'] for name, column in columns.items()})
strategy = calibrate_volatility_management(original_history, variance_history,
                                           calibration_months=CALIBRATION_MONTHS)
original, managed = strategy['original'], strategy['managed']
weights, scale = strategy['weights'], strategy['scale']
calibration_dates, evaluation_dates = strategy['calibration_index'], strategy['evaluation_index']
assert len(calibration_dates) == 120 and calibration_dates[-1] < evaluation_dates[0]
np.testing.assert_allclose(weights, strategy['lagged_measure'].rdiv(1).mul(scale))
np.testing.assert_allclose(managed, weights * original)
raw_calibration = original_history.div(variance_history.reindex(original_history.index).shift(1)).loc[calibration_dates]
np.testing.assert_allclose(raw_calibration.mul(scale).std(), original_history.loc[calibration_dates].std())
benchmarks = data.loc[evaluation_dates]
assert benchmarks[['mkt_rf', 'smb', 'hml', 'ff5_mkt_rf', 'ff5_smb', 'ff5_hml', 'rmw', 'cma', 'mom', 'rf']].notna().all().all()
print(f'Calibration: {calibration_dates[0]:%b %Y}–{calibration_dates[-1]:%b %Y} ({len(calibration_dates)} months).')
print(f'Out-of-sample evaluation: {evaluation_dates[0]:%b %Y}–{evaluation_dates[-1]:%b %Y} ({len(evaluation_dates)} months).')

### 2.1 Volatility clustering

The time-series plots show monthly factor returns over the evaluation period. Clusters of large movements illustrate why recent realized variance may contain information about next month's risk. The strategy uses this risk measure without forecasting the direction of returns.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), sharex=True, layout='constrained')
for ax, name in zip(axes.flat, original):
    ax.plot(original.index, 100 * original[name], color=BLUE, linewidth=.7)
    ax.axhline(0, color=GRAY, linewidth=.6)
    ax.set(title=name, ylabel='Monthly return (%)', xlabel='')
plt.show()

The return histories contain both calm periods and clusters of large movements. These plots illustrate volatility clustering; the performance comparison below examines whether adjusting exposure using past risk improves the resulting investment.

### 2.2 Managed and unmanaged performance

I first compare Sharpe ratios and regress each managed payoff on its own unmanaged factor:

$$F_t^{VM}=\alpha+\beta F_t+\varepsilon_t.$$

The alpha measures average payoff beyond a constant exposure to that factor. The loading describes co-movement, with one as the reference for a one-for-one relationship. Error bars show approximate 95% confidence intervals using Newey–West standard errors with twelve monthly lags.

In [ ]:
own_rows = []
for name in original:
    fit = time_series_regression(managed[name], original[[name]], cov_type='HAC', hac_lags=12)
    own_rows.append({'Factor': name, 'Unmanaged SR': sharpe_ratio(original[name]),
        'Managed SR': sharpe_ratio(managed[name]),
        'Alpha': 100 * fit.params['const'], 'Alpha SE': 100 * fit.bse['const'],
        'Alpha t': fit.tvalues['const'], 'Loading': fit.params[name],
        'Loading SE': fit.bse[name], 'Loading t': fit.tvalues[name], 'Months': int(fit.nobs)})
performance = pd.DataFrame(own_rows).set_index('Factor')
assert performance.Months.eq(len(evaluation_dates)).all()
x = np.arange(len(performance))
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), layout='constrained')
performance[['Unmanaged SR', 'Managed SR']].rename(
    columns={'Unmanaged SR': 'Unmanaged', 'Managed SR': 'Managed'}).plot.bar(
    ax=axes[0], color=[GRAY, BLUE], rot=0, width=.75)
axes[0].set(title='Annualized Sharpe ratios', ylabel='Sharpe ratio', xlabel='')
axes[0].axhline(0, color=GRAY, linewidth=.7)
axes[0].legend(frameon=False)
axes[1].bar(x, performance.Alpha, yerr=1.96 * performance['Alpha SE'], color=TEAL, capsize=3)
axes[1].axhline(0, color=GRAY, linewidth=.7)
axes[1].set(title='Own-factor alphas', ylabel='Alpha (% monthly)', xticks=x, xticklabels=performance.index)
axes[2].bar(x, performance.Loading, yerr=1.96 * performance['Loading SE'], color=AMBER, capsize=3)
axes[2].axhline(1, color=NAVY, linewidth=.9, linestyle='--')
axes[2].set(title='Own-factor loadings', ylabel='Loading', xticks=x, xticklabels=performance.index)
plt.show()

Momentum shows the largest Sharpe-ratio improvement, while profitability also improves. HML and CMA do not improve on that measure. Only RMW and MOM have own-factor alpha intervals entirely above zero. The loadings are below one, confirming that these managed strategies do not move one for one with the original factors; this alone does not establish better performance.


#### Cumulative returns

The next plots follow one unit of capital through the evaluation period. I add RF to each excess payoff before compounding, using the same financing convention as in 04. No out-of-sample risk matching is imposed: equal calibration volatility need not remain equal afterward. These paths exclude trading and borrowing costs.

In [ ]:
# Funded total return = RF + the excess payoff, with no further volatility matching.
funded_original = original.add(benchmarks.rf, axis=0)
funded_managed = managed.add(benchmarks.rf, axis=0)
assert funded_original.gt(-1).all().all() and funded_managed.gt(-1).all().all()
wealth_original = (1 + funded_original).cumprod()
wealth_managed = (1 + funded_managed).cumprod()
initial_date = calibration_dates[-1]
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True, layout='constrained')
for ax, name in zip(axes.flat, original):
    pair = pd.concat([wealth_original[name], wealth_managed[name]], axis=1, keys=['Unmanaged', 'Managed'])
    pair = pd.concat([pd.DataFrame(1., index=[initial_date], columns=pair.columns), pair])
    pair.plot(ax=ax, logy=True, color=[GRAY, BLUE], linewidth=1.4)
    ax.set(title=name, xlabel='', ylabel='Growth of one unit (log scale)')
    ax.legend(frameon=False, fontsize=9)
plt.show()

Managed momentum finishes above its unmanaged counterpart, while the other managed paths finish below theirs. RMW illustrates why the measures need not agree: its Sharpe ratio improves, but its funded growth is lower. The scale fixed during calibration affects the cumulative result, and out-of-sample volatility is not matched again.


### 2.3 Asset-pricing performance

I then compare each managed factor with CAPM, FF3, FF5 and FF5 augmented with ordinary momentum:

$$F_t^{VM}=\alpha+\boldsymbol\beta'\mathbf F_t+\varepsilon_t.$$

The bars show monthly alphas with approximate Newey–West 95% confidence intervals, again using twelve lags. All four models use the same out-of-sample months. FF3 and FF5 retain their separately published factor definitions. Adding MOM asks whether managed momentum contributes beyond constant exposure to ordinary momentum and FF5.

In [ ]:
models = {'CAPM': ['mkt_rf'], 'FF3': ['mkt_rf', 'smb', 'hml'],
    'FF5': ['ff5_mkt_rf', 'ff5_smb', 'ff5_hml', 'rmw', 'cma'],
    'FF5 + MOM': ['ff5_mkt_rf', 'ff5_smb', 'ff5_hml', 'rmw', 'cma', 'mom']}
model_rows = []
for name in managed:
    for model, regressors in models.items():
        fit = time_series_regression(managed[name], benchmarks[regressors], cov_type='HAC', hac_lags=12)
        model_rows.append({'Factor': name, 'Model': model,
            'Alpha': 100 * fit.params['const'], 'Alpha SE': 100 * fit.bse['const'],
            'Alpha t': fit.tvalues['const'], 'Months': int(fit.nobs)})
model_results = pd.DataFrame(model_rows).set_index(['Factor', 'Model'])
assert model_results.Months.eq(len(evaluation_dates)).all()
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True, layout='constrained')
x = np.arange(len(models))
for ax, name in zip(axes.flat, managed):
    values = model_results.loc[name].reindex(models)
    ax.bar(x, values.Alpha, yerr=1.96 * values['Alpha SE'], capsize=3, color=[NAVY, BLUE, TEAL, AMBER])
    ax.axhline(0, color=GRAY, linewidth=.8)
    ax.set(title=name, ylabel='Alpha (% monthly)', xticks=x,
           xticklabels=['CAPM', 'FF3', 'FF5', 'FF5 +\nMOM'], xlabel='')
plt.show()

Managed momentum retains positive alpha under all four benchmarks. Adding ordinary MOM reduces its alpha to about 0.475% per month, but the confidence interval remains above zero. RMW and CMA also retain positive alphas under FF5 + MOM, while MKT and SMB do not reject zero alpha. These are out-of-sample results for the specified rule, before costs and leverage constraints, rather than guarantees of future performance.


## References

- Moreira, A., and Muir, T. (2017). Volatility-managed portfolios. *The Journal of Finance*, 72(4), 1611–1644. [DOI](https://doi.org/10.1111/jofi.12513).
- Kenneth R. French Data Library. [Factor and portfolio data](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html).
